# Lab 6: End-to-End Eval Pipeline + Production Observability (Opik)
## CCI Session 7

**Duration:** 25 minutes

### Clinical Scenario
> The Oncology Summary Assistant (Wilms tumor focus) is about to be piloted on the pediatric oncology floor. Before it sees real patient cases, the AI Office needs (1) a single eval pipeline that combines everything from Labs 2-4 into one CI-runnable artifact, and (2) a production observability layer so we can detect drift the moment a prompt change degrades clinical quality.

### Objective
- Integrate the labeled dataset (Lab 2), functional metric (Lab 3), and validated LLM-judge (Lab 4) into a single `run_evals(...)` function
- Make the eval harness behave like a CI step (exit 0/1 against thresholds)
- Wire production traces into Opik (Comet's open-source LLMOps platform) and watch a deliberately-degraded prompt show up as a drift signal
- Hand off a CI-ready harness for Session 9

This lab is the wrap-up: the eval harness is part of the product, not a side script.

## Cell 1 - Setup

We need: `openai` (model calls), `deepeval` (LLM-judge plumbing from Lab 4), `scikit-learn` + `pandas` (metric aggregation), `opik` (production tracing), `jsonschema` (functional schema check).

In [ ]:
!pip install -q openai deepeval scikit-learn pandas opik jsonschema

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
# Free Comet account required: https://www.comet.com/signup -> API Keys
os.environ['COMET_API_KEY'] = userdata.get('COMET_API_KEY')
os.environ['OPIK_API_KEY'] = userdata.get('COMET_API_KEY')  # opik reads this name too
os.environ['OPIK_WORKSPACE'] = userdata.get('COMET_WORKSPACE')  # your Comet username
os.environ['OPIK_PROJECT_NAME'] = 'oncology-summary-assistant'

## Cell 2 - Risk surface inventory

**Discussion (write your answer in this cell before continuing):** what could go wrong with the Oncology Summary Assistant in production? Pick your top 3 failure modes and tag each as either *functional* (something a deterministic check can catch) or *semantic* (needs a judge or human).

Strawman to react to:
1. Missing required section (e.g., no `Stage` field) - functional, schema check.
2. Hallucinated histology (says "favorable" when input was "diffuse anaplastic") - semantic, LLM-judge.
3. Recommendation drift after a silent prompt edit (e.g., someone removes "include all clinical findings") - operational, needs production observability.

Lab 6 builds one defensive layer per failure mode.

## Cell 3 - Implement the Oncology Summary Assistant

Single function: take a free-text patient case, return a structured Wilms-tumor-focused summary with five required sections.

In [ ]:
from openai import OpenAI
client = OpenAI()

SYSTEM_PROMPT_V1 = '''You are an oncology summary assistant for pediatric Wilms tumor cases at KHCC.
Given a patient case, produce a concise structured summary with EXACTLY these five sections, each on its own line and prefixed with the section name and a colon:

Diagnosis: <one line>
Stage: <I, II, III, IV, or V (bilateral)>
Histology: <favorable, focal anaplastic, or diffuse anaplastic>
Recommendation: <chemotherapy regimen, surgery, radiation as appropriate per COG/SIOP guidelines>
Monitoring: <key labs, imaging cadence, toxicity surveillance>

Rules:
- Include ALL clinical findings present in the case (do not omit lab values, imaging findings, or comorbidities).
- If a field cannot be determined from the input, write "unknown" rather than guessing.
- Never invent dosing; defer to standard COG regimens (EE-4A, DD4A, Regimen M, UH-1) by name.
'''

def oncology_summary(patient_case: str, system_prompt: str = SYSTEM_PROMPT_V1) -> str:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': patient_case},
        ],
    )
    return resp.choices[0].message.content

# Smoke test
demo = oncology_summary('4 yo F, 8cm right renal mass, no metastases, favorable histology on biopsy.')
print(demo)

## Cell 4 - Load the 30-case labeled dataset (from Lab 2)

Inline for self-containment. Each case has an input narrative and the expected `Stage` / `Histology` / `Recommendation` keywords we expect to see.

In [ ]:
TEST_CASES = [
    {'id': 1,  'input': '3 yo M, left 6cm renal mass confined to kidney, completely resected, favorable histology, negative nodes.',
     'expected_stage': 'I',   'expected_histology': 'favorable',         'expected_keywords': ['EE-4A', 'vincristine', 'dactinomycin']},
    {'id': 2,  'input': '2 yo F, right 4cm renal mass, weight 480g, FH, complete resection, age <2.',
     'expected_stage': 'I',   'expected_histology': 'favorable',         'expected_keywords': ['nephrectomy', 'observation']},
    {'id': 3,  'input': '5 yo M, 10cm right renal mass extending beyond kidney capsule but completely excised, FH, nodes negative.',
     'expected_stage': 'II',  'expected_histology': 'favorable',         'expected_keywords': ['EE-4A', 'vincristine', 'dactinomycin']},
    {'id': 4,  'input': '6 yo F, 9cm left renal mass with positive hilar lymph nodes, FH, no distant disease.',
     'expected_stage': 'III', 'expected_histology': 'favorable',         'expected_keywords': ['DD4A', 'doxorubicin', 'radiation']},
    {'id': 5,  'input': '4 yo M, 12cm renal mass with intraoperative tumor spill, FH, complete resection.',
     'expected_stage': 'III', 'expected_histology': 'favorable',         'expected_keywords': ['DD4A', 'flank radiation']},
    {'id': 6,  'input': '7 yo F, right renal mass with bilateral pulmonary nodules on CT chest, FH on biopsy.',
     'expected_stage': 'IV',  'expected_histology': 'favorable',         'expected_keywords': ['DD4A', 'Regimen M', 'lung']},
    {'id': 7,  'input': '3 yo M, bilateral renal masses, FH on biopsy, no metastases.',
     'expected_stage': 'V',   'expected_histology': 'favorable',         'expected_keywords': ['nephron-sparing', 'neoadjuvant']},
    {'id': 8,  'input': '5 yo F, 8cm left renal mass, diffuse anaplastic histology, stage III with positive nodes.',
     'expected_stage': 'III', 'expected_histology': 'diffuse anaplastic','expected_keywords': ['UH-1', 'cyclophosphamide', 'radiation']},
    {'id': 9,  'input': '4 yo M, 7cm right renal mass, focal anaplastic histology, stage I, complete resection.',
     'expected_stage': 'I',   'expected_histology': 'focal anaplastic',  'expected_keywords': ['EE-4A']},
    {'id': 10, 'input': '6 yo F, 11cm renal mass with IVC thrombus extending to right atrium, FH.',
     'expected_stage': 'III', 'expected_histology': 'favorable',         'expected_keywords': ['DD4A', 'cardiac', 'thrombus']},
    {'id': 11, 'input': '2 yo M, 5cm FH renal mass, stage II completely resected, ALT 80, AST 95.',
     'expected_stage': 'II',  'expected_histology': 'favorable',         'expected_keywords': ['EE-4A', 'hepatic']},
    {'id': 12, 'input': '8 yo F, 9cm right renal mass with single pulmonary nodule, FH, complete local resection.',
     'expected_stage': 'IV',  'expected_histology': 'favorable',         'expected_keywords': ['DD4A', 'lung']},
    {'id': 13, 'input': '3 yo M, 6cm FH stage I tumor, weight 600g, age >2.',
     'expected_stage': 'I',   'expected_histology': 'favorable',         'expected_keywords': ['EE-4A']},
    {'id': 14, 'input': '5 yo F, 10cm renal mass with diffuse anaplastic histology, stage IV with lung metastases.',
     'expected_stage': 'IV',  'expected_histology': 'diffuse anaplastic','expected_keywords': ['UH-2', 'carboplatin', 'etoposide']},
    {'id': 15, 'input': '4 yo M, FH stage II, intraoperative spill noted, complete resection.',
     'expected_stage': 'III', 'expected_histology': 'favorable',         'expected_keywords': ['DD4A', 'flank radiation']},
    {'id': 16, 'input': '6 yo F, FH stage III with positive margins.',
     'expected_stage': 'III', 'expected_histology': 'favorable',         'expected_keywords': ['DD4A', 'radiation']},
    {'id': 17, 'input': '7 yo M, 8cm FH stage I, complete resection, age 7.',
     'expected_stage': 'I',   'expected_histology': 'favorable',         'expected_keywords': ['EE-4A']},
    {'id': 18, 'input': '5 yo F, FH stage IV with liver metastases.',
     'expected_stage': 'IV',  'expected_histology': 'favorable',         'expected_keywords': ['DD4A', 'Regimen M']},
    {'id': 19, 'input': '3 yo M, focal anaplastic stage III with positive nodes.',
     'expected_stage': 'III', 'expected_histology': 'focal anaplastic',  'expected_keywords': ['DD4A', 'radiation']},
    {'id': 20, 'input': '4 yo F, bilateral Wilms tumors with diffuse anaplastic histology on biopsy.',
     'expected_stage': 'V',   'expected_histology': 'diffuse anaplastic','expected_keywords': ['UH-1', 'neoadjuvant']},
    {'id': 21, 'input': '2 yo M, FH stage II with negative nodes and clear margins.',
     'expected_stage': 'II',  'expected_histology': 'favorable',         'expected_keywords': ['EE-4A']},
    {'id': 22, 'input': '6 yo F, FH stage III with peritoneal seeding.',
     'expected_stage': 'III', 'expected_histology': 'favorable',         'expected_keywords': ['DD4A', 'whole abdomen radiation']},
    {'id': 23, 'input': '5 yo M, FH stage I, age 4, weight 700g, complete resection.',
     'expected_stage': 'I',   'expected_histology': 'favorable',         'expected_keywords': ['EE-4A']},
    {'id': 24, 'input': '4 yo F, diffuse anaplastic stage II.',
     'expected_stage': 'II',  'expected_histology': 'diffuse anaplastic','expected_keywords': ['UH-1', 'doxorubicin']},
    {'id': 25, 'input': '3 yo M, FH stage IV with lung and bone lesions.',
     'expected_stage': 'IV',  'expected_histology': 'favorable',         'expected_keywords': ['DD4A', 'Regimen M', 'radiation']},
    {'id': 26, 'input': '7 yo F, FH stage III with biopsy prior to nephrectomy.',
     'expected_stage': 'III', 'expected_histology': 'favorable',         'expected_keywords': ['DD4A', 'radiation']},
    {'id': 27, 'input': '5 yo M, FH stage II, complete resection, no nodes positive.',
     'expected_stage': 'II',  'expected_histology': 'favorable',         'expected_keywords': ['EE-4A']},
    {'id': 28, 'input': '4 yo F, focal anaplastic stage IV with lung mets.',
     'expected_stage': 'IV',  'expected_histology': 'focal anaplastic',  'expected_keywords': ['DD4A', 'lung']},
    {'id': 29, 'input': '6 yo M, FH stage III with cytology-positive ascites.',
     'expected_stage': 'III', 'expected_histology': 'favorable',         'expected_keywords': ['DD4A', 'whole abdomen radiation']},
    {'id': 30, 'input': '2 yo F, FH stage I, weight 520g, age <2.',
     'expected_stage': 'I',   'expected_histology': 'favorable',         'expected_keywords': ['nephrectomy', 'observation']},
]
print(f'{len(TEST_CASES)} test cases loaded')

## Cell 5 - Functional metric: schema check

**TODO:** Implement `eval_schema(output: str) -> bool`. Return `True` only if the output contains all five required sections (`Diagnosis:`, `Stage:`, `Histology:`, `Recommendation:`, `Monitoring:`), each on its own line with a non-empty value.

Hint: a simple regex-per-section approach works. You don't need full JSON Schema for this one - we are validating a plain-text contract.

In [ ]:
import re

REQUIRED_SECTIONS = ['Diagnosis', 'Stage', 'Histology', 'Recommendation', 'Monitoring']

def eval_schema(output: str) -> bool:
    # TODO: return True only if every section in REQUIRED_SECTIONS is present
    # AND has a non-empty value after the colon on the same line.
    # Hint: re.search(rf'^{section}:\s*\S.*', output, flags=re.MULTILINE)
    ...

# Quick check
assert eval_schema(demo) in (True, False)  # not None
print('Schema check on smoke-test output:', eval_schema(demo))

## Cell 6 - LLM-judge metric: clinical relevance (validated v2 from Lab 4)

Returns a 1-5 score plus a severity tag. We use a structured-output prompt so the score is parseable. Threshold: a score `<= 2` is treated as a **high-severity** failure (clinically misleading).

In [ ]:
import json

JUDGE_SYSTEM = '''You are a pediatric oncology attending grading an AI-generated Wilms tumor case summary.
Score the summary against the input case on CLINICAL RELEVANCE (1-5):
  5 = all clinical findings captured, stage and histology correct, recommendation matches COG/SIOP guideline
  4 = minor omission, stage and histology correct, recommendation reasonable
  3 = noticeable omission OR weak recommendation, but not unsafe
  2 = wrong stage, wrong histology, or unsafe recommendation
  1 = grossly wrong, would harm patient if followed
Return ONLY a JSON object: {"score": <int 1-5>, "severity": "low"|"medium"|"high", "reason": "<one sentence>"}
Severity "high" iff score <= 2.
'''

def judge_clinical_relevance(case_input: str, summary: str) -> dict:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': JUDGE_SYSTEM},
            {'role': 'user', 'content': f'CASE:\n{case_input}\n\nSUMMARY:\n{summary}'},
        ],
    )
    try:
        out = json.loads(resp.choices[0].message.content)
        out['score'] = int(out.get('score', 0))
    except Exception:
        out = {'score': 0, 'severity': 'high', 'reason': 'judge parse failure'}
    return out

print(judge_clinical_relevance(TEST_CASES[0]['input'], demo))

## Cell 7 - Wrap evals as `run_evals(test_cases) -> dict`

**TODO:** This is the heart of the lab. Implement `run_evals` so it:
1. Iterates the test cases, calls `oncology_summary`, applies `eval_schema` and `judge_clinical_relevance`.
2. Aggregates: `schema_pass_rate`, `judge_mean_score`, `judge_high_severity_failures`.
3. Compares each metric against the THRESHOLDS dict.
4. Returns the metrics dict AND prints/returns an exit code: `0` if all pass, `1` if any fail.

This is what a CI step would call.

In [ ]:
import sys
import pandas as pd

THRESHOLDS = {
    'schema_pass_rate':           0.90,   # >=
    'judge_mean_score':           4.0,    # >=
    'judge_high_severity_failures': 1,    # <= (at most 1)
}

def run_evals(test_cases, system_prompt=SYSTEM_PROMPT_V1, verbose=True):
    rows = []
    for tc in test_cases:
        # TODO: 1. call oncology_summary(tc['input'], system_prompt=system_prompt)
        # TODO: 2. eval_schema(...) -> bool
        # TODO: 3. judge_clinical_relevance(tc['input'], summary) -> dict
        # TODO: 4. append a row dict with id, schema_ok, judge_score, severity
        ...
    df = pd.DataFrame(rows)
    metrics = {
        'schema_pass_rate':            float(df['schema_ok'].mean()),
        'judge_mean_score':            float(df['judge_score'].mean()),
        'judge_high_severity_failures': int((df['severity'] == 'high').sum()),
    }
    # TODO: compare against THRESHOLDS; set exit_code = 0 if all pass else 1
    exit_code = ...
    if verbose:
        print('Metrics:', metrics)
        print('Exit code:', exit_code)
    return metrics, exit_code, df

baseline_metrics, baseline_exit, baseline_df = run_evals(TEST_CASES)

## Cell 8 - CI integration preview (Session 9)

In Session 9 you will move this into your repo. The shape of `.github/workflows/eval.yml` is:

```yaml
name: eval
on: [push, pull_request]
jobs:
  evals:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.11' }
      - run: pip install -r requirements.txt
      - env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
          COMET_API_KEY:  ${{ secrets.COMET_API_KEY }}
        run: python run_evals.py   # exits 1 if thresholds breached -> PR fails
```

`run_evals.py` is essentially Cell 7 wrapped in `if __name__ == '__main__': sys.exit(run_evals(TEST_CASES)[1])`. A failing eval blocks the merge - the harness IS the product.

## Cell 9 - Set up Opik tracing

Opik is Comet's open-source LLMOps tool. We initialize the client, then decorate `oncology_summary` with `@opik.track` so every call is logged as a trace with input/output/latency/tokens. Traces show up in your Opik project at https://www.comet.com/opik/.

**Note:** the Opik SDK evolves quickly - check https://www.comet.com/docs/opik/ if any of these symbols don't exist on your installed version.

**TODO:** Wire `@opik.track` onto `oncology_summary` (or a thin wrapper) and re-run a single case so you see a trace in the dashboard.

In [ ]:
import opik
from opik import opik_context

# Configure once. Reads OPIK_API_KEY / OPIK_WORKSPACE / OPIK_PROJECT_NAME from env.
opik.configure(use_local=False)

# TODO: define a traced wrapper. Example pattern:
#
# @opik.track(project_name=os.environ['OPIK_PROJECT_NAME'])
# def traced_oncology_summary(patient_case: str, system_prompt: str = SYSTEM_PROMPT_V1) -> str:
#     out = oncology_summary(patient_case, system_prompt)
#     opik_context.update_current_trace(
#         tags=['lab6', 'baseline'],
#         metadata={'prompt_version': 'v1'},
#     )
#     return out

traced_oncology_summary = ...
_ = traced_oncology_summary(TEST_CASES[0]['input'])

## Cell 10 - Push 5 synthetic production traces

Simulates a quiet morning on the floor: 5 cases come through the assistant. After this cell runs, open the Opik dashboard - you should see 5 traces under project `oncology-summary-assistant`, each with input, output, latency, and the `prompt_version: v1` metadata tag.

In [ ]:
for tc in TEST_CASES[:5]:
    _ = traced_oncology_summary(tc['input'])
print('5 production traces sent. Open Opik dashboard to inspect.')

## Cell 11 - Drift detection demo

Now we deliberately degrade the system prompt - we remove the "include ALL clinical findings" instruction and the "Never invent dosing" guardrail. In the real world this might happen because someone trims the prompt to save tokens, or refactors and loses a line.

**TODO:** Define `SYSTEM_PROMPT_V2_DEGRADED` (drop those two rules). Re-run `run_evals` with it. Report the before/after numbers.

In [ ]:
# TODO: build the degraded prompt by stripping the two key rules.
SYSTEM_PROMPT_V2_DEGRADED = '''You are an oncology summary assistant for pediatric Wilms tumor cases.
Given a patient case, produce a short summary covering diagnosis, stage, histology, recommendation, and monitoring.
Be concise.
'''

# TODO: also re-decorate so the new traces are tagged 'degraded' / prompt_version 'v2'.
# Example:
# @opik.track(project_name=os.environ['OPIK_PROJECT_NAME'])
# def traced_oncology_summary_v2(patient_case):
#     out = oncology_summary(patient_case, SYSTEM_PROMPT_V2_DEGRADED)
#     opik_context.update_current_trace(tags=['lab6', 'degraded'], metadata={'prompt_version': 'v2'})
#     return out

# Push 5 degraded traces to Opik for the dashboard
# for tc in TEST_CASES[:5]:
#     traced_oncology_summary_v2(tc['input'])

# TODO: re-run the eval harness with the degraded prompt
degraded_metrics, degraded_exit, degraded_df = run_evals(TEST_CASES, system_prompt=SYSTEM_PROMPT_V2_DEGRADED)

print('--- Drift report ---')
print(f'schema_pass_rate:          {baseline_metrics["schema_pass_rate"]:.2f}  ->  {degraded_metrics["schema_pass_rate"]:.2f}')
print(f'judge_mean_score:          {baseline_metrics["judge_mean_score"]:.2f}  ->  {degraded_metrics["judge_mean_score"]:.2f}')
print(f'high_severity_failures:    {baseline_metrics["judge_high_severity_failures"]}     ->  {degraded_metrics["judge_high_severity_failures"]}')
print(f'CI exit code:              {baseline_exit}     ->  {degraded_exit}')

## Cell 12 - Stretch: Opik online metric for auto-scoring every production trace

**TODO:** Wire `eval_schema` (or the LLM judge) as an Opik *online* metric - i.e. every trace that lands in production gets scored automatically and you alert when the rolling pass rate drops below threshold. This is what turns drift detection from manual into actual observability.

Pattern (check current Opik docs - the API name is `opik.evaluation` / `Opik().log_traces_feedback_scores`):

```python
from opik import Opik
client_opik = Opik()
# After traced_oncology_summary returns, attach a feedback score:
# client_opik.log_traces_feedback_scores([
#     {'id': trace_id, 'name': 'schema_ok', 'value': 1.0 if eval_schema(out) else 0.0}
# ])
```

Bonus: set up an Opik dashboard alert when `schema_ok` rolling-mean over 50 traces drops below 0.9.

## Cell 13 - Hand-off to Session 9

**Take-home (full spec in `session_7_curriculum.md`):**
1. Lift `oncology_summary`, `eval_schema`, `judge_clinical_relevance`, `run_evals`, and `TEST_CASES` out of this notebook into a tiny Python package: `oncology_assistant/{model.py, evals.py, data.py}` plus `run_evals.py` at the repo root.
2. Add a `requirements.txt` and a basic `README.md`.
3. In Session 9 you will: (a) commit it to a fresh GitHub repo, (b) add the `.github/workflows/eval.yml` from Cell 8, (c) configure repo secrets for `OPENAI_API_KEY` and `COMET_API_KEY`, (d) open a PR that intentionally degrades the prompt - watch the eval check fail and block the merge.

That moment is the deliverable: an eval harness that prevents a clinical-quality regression from ever reaching production. The harness IS the product.

## Cell 14 - KHCC Connection

At KHCC, an Oncology Summary Assistant that silently regresses is more dangerous than no assistant at all - clinicians stop scrutinizing outputs once they trust the tool. The eval pipeline you built today is what gives the AI Office authority to say "yes, ship this prompt change" or "no, this drops clinical-relevance from 4.3 to 3.1, fix it first." Opik is what tells you the same thing in production, before a chart-review committee finds it the hard way.